CASE STUDY 03 - COMPUTER VISION

Classifying Fruits and Vegetables Using Image Classification
The given dataset contains images of a variety of fruits and vegetables, offering a rich source for
developing and testing image recognition algorithms. The food items are categorized as follows:
Fruits:
Banana, Apple, Pear, Grapes, Orange, Kiwi, Watermelon, Pomegranate, Pineapple, Mango
Vegetables:
Cucumber, Carrot, Capsicum, Onion, Potato, Lemon, Tomato, Radish, Beetroot, Cabbage,
Lettuce, Spinach, Soybean, Cauliflower, Bell Pepper, Chilly, Pepper, Turnip, Corn, Sweetcorn,
Sweet Potato, Paprika, Jalapeño, Ginger, Garlic, Peas, Eggplant
Given this dataset, your task is to create a machine learning model that can classify the images
into two main categories: Fruits and Vegetables

Load the data from google drive as data is large

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
# Copy from Drive to Colab
!cp "/content/drive/MyDrive/ICT_AIML/datasetict.zip" /content/dataset.zip

In [5]:
# unzip the data
!unzip -q /content/dataset.zip -d /content/raw_data

In [6]:
# contents of dataset
!ls /content/raw_data

test  train  validation


Labelling Fruits and Vegetables

In [7]:
import os
import shutil
import tensorflow as tf
from PIL import Image
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input

# 1. Define Categories
fruits_list = ['Banana', 'Apple', 'Pear', 'Grapes', 'Orange', 'Kiwi', 'Watermelon', 'Pomegranate', 'Pineapple', 'Mango']
raw_data_path = '/content/raw_data'
final_data_path = '/content/dataset_final'

# 2. Organize into Binary Folders (Fruits vs Vegetables)
for split in ['train', 'test', 'validation']:
    for cat in ['Fruits', 'Vegetables']:
        os.makedirs(os.path.join(final_data_path, split, cat), exist_ok=True)

    current_split_path = os.path.join(raw_data_path, split)
    for folder in os.listdir(current_split_path):
        source_folder = os.path.join(current_split_path, folder)
        if not os.path.isdir(source_folder): continue

        label = 'Fruits' if folder in fruits_list else 'Vegetables'
        dest = os.path.join(final_data_path, split, label)

        # Many web images (especially PNGs) use a mode called P (Palette). Instead of storing actual colors for every pixel, they store a "map" of 256 colors. so converting to RGBA
        for img_name in os.listdir(source_folder):
            # THE PIL FIX: Clean transparency/palettes while moving
            try:
                with Image.open(os.path.join(source_folder, img_name)) as img:
                    clean_img = img.convert('RGBA').convert('RGB')
                    clean_img.save(os.path.join(dest, f"{folder}_{img_name}"))
            except:
                continue

print("Data is organized")

Data is organized


Load training and validation data

In [8]:
# Load datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/dataset_final/train',
    image_size=(224, 224),
    batch_size=32,
    label_mode='binary'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/dataset_final/validation',
    image_size=(224, 224),
    batch_size=32,
    label_mode='binary'
)

# Optimization: Prefetching keeps the GPU fed with data
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

Found 3115 files belonging to 2 classes.
Found 351 files belonging to 2 classes.


Build  MobileNetV2 Model

In [9]:
# 1. Base Model
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

# 2. Final Model Architecture
model = models.Sequential([
    # This layer handles the -1 to 1 scaling MobileNetV2 needs
    layers.Lambda(preprocess_input, input_shape=(224, 224, 3)),

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2), # Prevents overfitting
    layers.Dense(1, activation='sigmoid') # 0 for Fruits, 1 for Vegetables
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/lambda_layer.py:65: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [10]:
# Training
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 255s 2s/step - accuracy: 0.9843 - loss: 0.0423 - val_accuracy: 1.0000 - val_loss: 0.0035
Epoch 2/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 217s 2s/step - accuracy: 1.0000 - loss: 0.0025 - val_accuracy: 1.0000 - val_loss: 0.0018
Epoch 3/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 278s 2s/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 1.0000 - val_loss: 0.0012
Epoch 4/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 235s 2s/step - accuracy: 1.0000 - loss: 9.3279e-04 - val_accuracy: 1.0000 - val_loss: 7.9926e-04
Epoch 5/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 215s 2s/step - accuracy: 1.0000 - loss: 6.6745e-04 - val_accuracy: 1.0000 - val_loss: 5.8978e-04


Test with Testing data

In [11]:
# Load the Test Dataset (just like Train and Val)
test_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/dataset_final/test',
    image_size=(224, 224),
    batch_size=32,
    label_mode='binary'
)

# Optimization
test_ds = test_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

#  Evaluate
print("Evaluating on the Test Set...")
test_loss, test_acc = model.evaluate(test_ds)
print(f"Final Test Accuracy: {test_acc * 100:.2f}%")

Found 359 files belonging to 2 classes.
Evaluating on the Test Set...
12/12 ━━━━━━━━━━━━━━━━━━━━ 25s 2s/step - accuracy: 1.0000 - loss: 5.8575e-04
Final Test Accuracy: 100.00%


The model has 100% accuracy as verified with trainig set data as well. The model can distinguish between fruits and vegetables.

Used Transfer learning build the model with MobileNetV2.